# Module 08 — Native tool calling: the regex is deleted

**THE ONE IDEA:** the tool call moves out of prose and into a **structured field the
provider guarantees**. Both regexes from module 07 are gone. Same question, same tools,
same answer — only the machinery changed.

Two things to watch, one per envelope:

| | OpenAI | Anthropic |
|---|---|---|
| stop signal flips to | `finish_reason = "tool_calls"` | `stop_reason = "tool_use"` |
| **the trap** | `message.content` is **`None`** | the call is a **block** inside `content[]` |
| result goes back as | a `role: "tool"` message | a `tool_result` block in a **user** message |


In [ ]:
import sys; sys.path.insert(0, "..")   # shared _providers.py / _tools.py at the phase root
import json
from _providers import get_client
from _tools import openai_schemas, anthropic_schemas, run_tool

QUESTION = ("What is the early repayment charge in year 2 on a 250000 loan? "
            "Look up the policy, then calculate it.")
TOOLS = ["search_policy", "calculate"]
print("no ACTION_RE. no FINAL_RE. that is the whole point.")

## The OpenAI-envelope loop

In [ ]:
def agent_openai(max_steps=6):
    client, model, _ = get_client("openai")
    messages = [{"role": "user", "content": QUESTION}]
    for step in range(1, max_steps + 1):
        r = client.chat.completions.create(model=model, max_tokens=500,
                                           tools=openai_schemas(TOOLS), messages=messages)
        msg = r.choices[0].message
        print(f"\nstep {step}: finish_reason={r.choices[0].finish_reason}  "
              f"content={msg.content!r}")            # <- None on a tool turn. THE TRAP.

        if r.choices[0].finish_reason != "tool_calls":
            return msg.content, step

        messages.append(msg)                          # the assistant turn, verbatim
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)  # a JSON *string* on the wire
            out = run_tool(tc.function.name, args)
            print(f"  -> {tc.function.name}({args}) = {out[:70]}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": out})
    return None, max_steps

ans_oa, steps_oa = agent_openai()
print("\nANSWER:", ans_oa)

## The Anthropic-envelope loop

Same logic, different shape. The tool call is a **block** you filter for, and the result
goes back inside a **user** message, not a `tool` role.

In [ ]:
def agent_anthropic(max_steps=6):
    client, model, _ = get_client("anthropic")
    messages = [{"role": "user", "content": QUESTION}]
    for step in range(1, max_steps + 1):
        r = client.messages.create(model=model, max_tokens=2000,
                                   tools=anthropic_schemas(TOOLS), messages=messages)
        print(f"\nstep {step}: stop_reason={r.stop_reason}  "
              f"blocks={[b.type for b in r.content]}")

        if r.stop_reason != "tool_use":
            return "".join(b.text for b in r.content if b.type == "text"), step

        messages.append({"role": "assistant", "content": r.content})
        results = []
        for b in r.content:
            if b.type == "tool_use":
                out = run_tool(b.name, b.input)
                print(f"  -> {b.name}({b.input}) = {out[:70]}")
                results.append({"type": "tool_result", "tool_use_id": b.id, "content": out})
        messages.append({"role": "user", "content": results})   # user, not "tool"
    return None, max_steps

ans_an, steps_an = agent_anthropic()
print("\nANSWER:", ans_an)

## Equivalence — the point of the ladder

In [ ]:
print("module 07 (regex, prose)   ->  10000")
print(f"module 08 openai envelope  ->  {steps_oa} steps")
print(f"module 08 anthropic envelope-> {steps_an} steps")
print()
print("LESSON — 07, 08 and 19 all produce the SAME answer. Only the machinery")
print("differs on screen. That equivalence is why this is a ladder.")
print()
print("What native tool calling actually bought you:")
print("  - the format is the provider's contract, not your regex")
print("  - a JSON Schema is enforced, so arg NAMES and TYPES are checked upstream")
print("  - the stop signal is explicit: finish_reason / stop_reason tells you to loop")
print("  - parallel calls become expressible (module 09)")
print()
print("What it did NOT buy you: the model can still pick the WRONG tool, loop")
print("forever, or keep going after it already has the answer. Schema validity is")
print("not judgment. Block D is about that gap.")

---

**Next:** `09_agent_parallel_tool_calls.ipynb`